# Task 1: Supervised learning 

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42

def set_seed(seed = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
         torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))# normal mnist normalization using mean 0.130and std 0.308 of the dataset
])

train_data = datasets.MNIST(root="./data", train=True , download=True, transform=transform)
test_data  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)



### Data split
The original training set is split into 80% training and 20% validation to perform model selection without contaminating the final test

In [ ]:
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size

split_generator = torch.Generator().manual_seed(SEED)    
## the train/validation split is always the same at every run, so
#I can be sure that the same samples are used for training and validation across different experiments, making the results comparable
train_subset , val_subset = random_split(train_data, [train_size, val_size], generator = split_generator)

print("Train subset:" , train_size)
print("Validation subset:" , val_size)
print("Test set:" , len(test_data))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:" , device)


Train subset: 48000
Validation subset: 12000
Test set: 10000
Running on: cpu


### Architectures
I compare two feed-forward models:

- ShallowNet: a simple baseline with one hidden layer.
- DeepNet: a deeper network with three hidden layers, batch normalization and dropout

I keep the shallow model intentionally simple, so the comparison with the deeper architecture is clean

In [6]:
class ShallowNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(784,128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128,10)

    def forward(self, x):
         x = x.view(x.size(0), -1)
         x = self.fc1(x)
         x = self.relu(x)
         x = self.fc2(x)
         return x


class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(784, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512,256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.fc4 = nn.Linear(128,10)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.view(x.size(0),-1)

        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc4(x)
        return x


### Training and evaluation functions


In [ ]:
def make_loader(dataset, batch_size, shuffle, seed=SEED):
    gen = None
    if shuffle:
        
        gen = torch.Generator().manual_seed(seed)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=gen)
    return loader


def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0


    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            bs = labels.size(0)
            total_loss = total_loss + loss.item() * bs

            _, pred = outputs.max(1)
            total = total + bs
            correct = correct + (pred == labels).sum().item()



    avg_loss = total_loss / total
    avg_acc = 100.0 * correct / total


    return avg_loss, avg_acc


def train_model(model, train_loader, val_loader, lr, epochs, weight_decay=0.0):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for e in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss = running_loss+loss.item() * bs

            _, pred =   outputs.max(1)
            total    =   total+bs
            correct =   correct+(pred ==labels).sum().item()

        train_losses.append(running_loss / total)
        train_accs.append(100.0*correct / total)

        if val_loader is not None:
            val_loss, val_acc= evaluate(model, val_loader, criterion)
            val_losses.append(val_loss)
            val_accs.append(val_acc)

    return train_losses,train_accs, val_losses, val_accs


def run_validation_experiment(model_class, batch_size, lr, epochs, weight_decay=0.0, seed=SEED):
    set_seed(seed)

    train_loader = make_loader(train_subset, batch_size=batch_size, shuffle=True, seed=seed)
    val_loader = make_loader(val_subset, batch_size=batch_size, shuffle=False, seed=seed)

    model = model_class()

    train_losses, train_accs, val_losses, val_accs = train_model(
        model,
        train_loader,
        val_loader,
        lr = lr,
        epochs = epochs,
        weight_decay = weight_decay     )

    out = {}
    out["train_losses"] = train_losses
    out["train_accs"] = train_accs
    out["val_losses"] = val_losses
    out["val_accs"] = val_accs
    out["best_val_acc"] = max(val_accs)
    out["final_val_acc"] = val_accs[-1]
    return out

def plot_curves(train_losses, train_accs, val_losses, val_accs):
    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_losses, label='Train loss')
    plt.plot(epochs, val_losses, label='Validation loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_accs, label='Train accuracy')
    plt.plot(epochs, val_accs, label='Validation accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()


### Hyperparameter search
I tune the two architectures separately on the training/validation split.

I test different parameters for a model, run a validation experiment for each, save the best validation accuracy and final validation accuracy, then select the learning rate with the best validation accuracy

I run it for only 10 epochs, since the quality of a hyperparameter can already be assessed without fully training the model. 10 epochs provide a good guess if that parameter is likely to perform well. It's a tradeoff between computational complexity and estimation 

In [ ]:
def tune_learning_rate(model_class, model_name, lrs_to_try, batch_size=64, epochs=10, weight_decay=0.0):
    print("Tuning learning rate for " + str(model_name))
    results = []

    for lr in lrs_to_try:
        result = run_validation_experiment(
            model_class,
            batch_size = batch_size,
            lr = lr,
            epochs = epochs,
            weight_decay = weight_decay,
            seed = SEED,
        )

        results.append((lr, result["best_val_acc"], result["final_val_acc"]))

        print("lr=" + str(lr) + " , best val acc=" + f"{result['best_val_acc']:.2f}%" +
            ", final val acc=" + f"{result['final_val_acc']:.2f}%"
        )

    best_lr = results[0][0]
    best_score = results[0][1]

    for item in results:
        if item[1] > best_score:
            best_score = item[1]
            best_lr = item[0]

    print("Best learning rate for " + str(model_name) + ": " + str(best_lr) + "\n")
    return best_lr, results


lrs_to_try = [0.1, 0.01, 0.001, 0.0001]

shallow_best_lr, shallow_lr_results = tune_learning_rate(ShallowNet, "ShallowNet", lrs_to_try)
deep_best_lr, deep_lr_results = tune_learning_rate(DeepNet, "DeepNet", lrs_to_try)


Tuning learning rate for ShallowNet
lr=0.1 , best val acc=22.60%, final val acc=20.82%
lr=0.01 , best val acc=95.20%, final val acc=95.20%
lr=0.001 , best val acc=97.38%, final val acc=97.36%
lr=0.0001 , best val acc=95.77%, final val acc=95.77%
Best learning rate for ShallowNet: 0.001

Tuning learning rate for DeepNet
lr=0.1 , best val acc=97.16%, final val acc=97.00%
lr=0.01 , best val acc=98.00%, final val acc=97.78%
lr=0.001 , best val acc=98.05%, final val acc=98.05%
lr=0.0001 , best val acc=97.90%, final val acc=97.90%
Best learning rate for DeepNet: 0.001



I now tune the batch size for each architecture using the learning rate selected above


In [ ]:
def tune_batch_size(model_class, model_name, batch_sizes, lr, epochs=10, weight_decay=0.0):
    print("Tuning batch size for " + str(model_name))
    results = []

    for bs in batch_sizes:
        result = run_validation_experiment(
            model_class,
            batch_size = bs,
            lr = lr,
            epochs = epochs,
            weight_decay = weight_decay,
            seed = SEED,
        )

        results.append((bs, result["best_val_acc"], result["final_val_acc"]))

        print("batch size=" + str(bs) +" , best val acc=" + f"{result['best_val_acc']:.2f}%" +
            ", final val acc=" + f"{result['final_val_acc']:.2f}%"
        )

    best_bs = results[0][0]
    best_score = results[0][1]

    for item in results:
        if item[1] > best_score:
            best_score = item[1]
            best_bs = item[0]

    print("Best batch size for " + str(model_name) + ": " + str(best_bs) + "\n")
    return best_bs, results


batch_sizes = [32, 64, 128, 256]

shallow_best_bs, shallow_bs_results = tune_batch_size(
ShallowNet, "ShallowNet", batch_sizes, lr = shallow_best_lr)
deep_best_bs, deep_bs_results = tune_batch_size(
DeepNet, "DeepNet", batch_sizes, lr = deep_best_lr)


Tuning batch size for ShallowNet
batch size=32 , best val acc=97.44%, final val acc=97.18%
batch size=64 , best val acc=97.38%, final val acc=97.36%
batch size=128 , best val acc=97.58%, final val acc=97.58%
batch size=256 , best val acc=97.19%, final val acc=97.18%
Best batch size for ShallowNet: 128

Tuning batch size for DeepNet
batch size=32 , best val acc=98.18%, final val acc=98.18%
batch size=64 , best val acc=98.05%, final val acc=98.05%
batch size=128 , best val acc=98.04%, final val acc=98.04%
batch size=256 , best val acc=98.07%, final val acc=98.07%
Best batch size for DeepNet: 32




Best batch size for ShallowNet is 128, anyway,
since the accuracy betweeen batchsize 32 and 128 for the shallow nn is almost identical I ll set the best batch size to 32 to save a lots of computational power for the next experiments.

Finally, I tune weight decay for  the deeper network using the best learning rate and batch size found so far.
since the objective of the experiment is to make comparison between a not regularized  model with one which is regularized, I ll do it only on the DeepNet model

weight decay intruduces regularization so this is why I don t tune it for the shallownet

I also set the number of epochs = 30. the triple of the other experiments because the overfitting could start later( and  we are setting a regularization parameter to avoid overfitting).  

Training with more updates has more probability to learn from not significant details not relevant to the task. ( this could likely happen in case of not regularized model, like shallownet ). Here, where i tune the DeepNet , which is already regularized by batchnorm and droput, setting epochs=30 or epochs=10 it shouldn t make any substantial differences. I ll use 30 just to stay coherent with my reasoning

In [ ]:
def tune_weight_decay(model_class, model_name, weight_decays, lr, batch_size, epochs=30):
    print("Tuning weight decay for " + str(model_name))
    results = []

    for wd in weight_decays:
        result = run_validation_experiment(
            model_class,
            batch_size = batch_size,
            lr = lr,
            epochs = epochs,
            weight_decay = wd,
            seed = SEED,
        )

        results.append((wd, result["best_val_acc"], result["final_val_acc"]))

        print("weight decay=" + str(wd) +
              " , best val acc=" + f"{result['best_val_acc']:.2f}%" +
            " , final val acc=" + f"{result['final_val_acc']:.2f}%"
        )

    best_wd = results[0][0]
    best_score = results[0][1]

    for item in results:
        if item[1] > best_score:
            best_score = item[1]
            best_wd = item[0]

    print("Best weight decay for " + str(model_name) + ": " + str(best_wd) + "\n")
    return best_wd, results


weight_decays = [0.0, 0.0001, 0.001, 0.01, 0.1]

deep_best_lr = 0.001
deep_best_bs = 32
deep_best_wd, deep_wd_results = tune_weight_decay(
DeepNet, "DeepNet", weight_decays, lr = deep_best_lr, batch_size = deep_best_bs)


Tuning weight decay for DeepNet
weight decay=0.0 , best val acc=98.50% , final val acc=98.49%
weight decay=0.0001 , best val acc=98.15% , final val acc=98.08%
weight decay=0.001 , best val acc=97.29% , final val acc=96.92%
weight decay=0.01 , best val acc=95.44% , final val acc=95.28%
weight decay=0.1 , best val acc=89.96% , final val acc=89.42%
Best weight decay for DeepNet: 0.0



weight decay is 0 so deepnet is regularized only through batch normalization and dropout


At this point, hyperparameter selection for the final models is  complete.


Next section should be interpreted as a qualitative side analysis, not as a step used to choose the final ShallowNet configuration.
now I wanna see if what I said it s true: ( about my previous statement "....the overfitting could start later( since we are setting a regularization parameter). Training with more updates has more probability to learn from not significant details not relevant to the task . this could be true in case of not regularization   ) " .
So I ll try to tune weigh decay for the shallow model (which is, in fact, not regularized at all throught batchnorm, dropout,...)with different epochs just to see if , augmenting the number of epochs, there could be a weight decay that improve performances, in particular in the case where the num of epochs is larger


(from this point until " final training stratgy" I just want to validate this idea/reasoning, it s not totally correlated with the objective of the experiment itsself)

In [ ]:

def tune_weight_decay(model_class, model_name, weight_decays, lr, batch_size, epochs=5):
    print("Tuning weight decay for " + str(model_name))
    results = []

    for wd in weight_decays:
        result = run_validation_experiment(
            model_class,
            batch_size = batch_size,
            lr = lr,
            epochs = epochs,
            weight_decay = wd,
            seed = SEED,
        )

        results.append((wd, result["best_val_acc"], result["final_val_acc"]))

        print("weight decay=" + str(wd) +
              " , best val acc=" + f"{result['best_val_acc']:.2f}%" +
            " , final val acc=" + f"{result['final_val_acc']:.2f}%"
        )

    best_wd = results[0][0]
    best_score = results[0][1]

    for item in results:
        if item[1] > best_score:
            best_score = item[1]
            best_wd = item[0]

    print("Best weight decay for " + str(model_name) + ": " + str(best_wd) + "\n")
    return best_wd, results


weight_decays = [0.0, 0.0001, 0.001, 0.01, 0.1]


shallow_best_wd, shallow_wd_results = tune_weight_decay(
    ShallowNet, "ShallowNet", weight_decays, lr = shallow_best_lr, batch_size = shallow_best_bs )


Tuning weight decay for ShallowNet
weight decay=0.0 , best val acc=97.12% , final val acc=96.85%
weight decay=0.0001 , best val acc=97.06% , final val acc=96.42%
weight decay=0.001 , best val acc=96.72% , final val acc=96.19%
weight decay=0.01 , best val acc=95.12% , final val acc=94.73%
weight decay=0.1 , best val acc=88.49% , final val acc=88.26%
Best weight decay for ShallowNet: 0.0



In [ ]:

weight_decays = [0.0, 0.0001, 0.001, 0.01, 0.1]


shallow_best_wd, shallow_wd_results = tune_weight_decay(
    ShallowNet, "ShallowNet", weight_decays, lr = shallow_best_lr, batch_size = shallow_best_bs, epochs=10)


Tuning weight decay for ShallowNet
weight decay=0.0 , best val acc=97.44% , final val acc=97.18%
weight decay=0.0001 , best val acc=97.58% , final val acc=97.08%
weight decay=0.001 , best val acc=97.06% , final val acc=96.78%
weight decay=0.01 , best val acc=95.42% , final val acc=94.47%
weight decay=0.1 , best val acc=89.16% , final val acc=89.16%
Best weight decay for ShallowNet: 0.0001



In [ ]:


weight_decays = [0.0, 0.0001, 0.001, 0.01, 0.1]


shallow_best_wd, shallow_wd_results = tune_weight_decay(
    ShallowNet, "ShallowNet", weight_decays, lr = shallow_best_lr, batch_size = shallow_best_bs, epochs=15)


Tuning weight decay for ShallowNet
weight decay=0.0 , best val acc=97.44% , final val acc=97.24%
weight decay=0.0001 , best val acc=97.58% , final val acc=97.30%
weight decay=0.001 , best val acc=97.15% , final val acc=96.85%
weight decay=0.01 , best val acc=95.53% , final val acc=95.27%
weight decay=0.1 , best val acc=89.16% , final val acc=88.93%
Best weight decay for ShallowNet: 0.0001



In [ ]:


weight_decays = [0.0, 0.0001, 0.001, 0.01, 0.1]


shallow_best_wd, shallow_wd_results = tune_weight_decay(
ShallowNet, "ShallowNet", weight_decays, lr = shallow_best_lr, batch_size = shallow_best_bs, epochs=25)


Tuning weight decay for ShallowNet
weight decay=0.0 , best val acc=97.63% , final val acc=97.63%
weight decay=0.0001 , best val acc=97.58% , final val acc=97.29%
weight decay=0.001 , best val acc=97.59% , final val acc=97.10%
weight decay=0.01 , best val acc=95.53% , final val acc=95.46%
weight decay=0.1 , best val acc=89.53% , final val acc=87.92%
Best weight decay for ShallowNet: 0.0



In [10]:

shallow_best_lr = 0.001
deep_best_lr = 0.001
deep_best_wd = 0.0
deep_best_bs = 32
shallow_best_bs = 32 #(128 actually)
weight_decays = [0.0, 0.0001, 0.001, 0.01, 0.1]


shallow_best_wd, shallow_wd_results = tune_weight_decay(
ShallowNet, "ShallowNet", weight_decays, lr = shallow_best_lr, batch_size = shallow_best_bs, epochs=50)


Tuning weight decay for ShallowNet
weight decay=0.0 , best val acc=97.76% , final val acc=97.54%
weight decay=0.0001 , best val acc=97.58% , final val acc=97.33%
weight decay=0.001 , best val acc=97.59% , final val acc=97.29%
weight decay=0.01 , best val acc=95.69% , final val acc=94.77%
weight decay=0.1 , best val acc=89.53% , final val acc=88.01%
Best weight decay for ShallowNet: 0.0



my experiments fails, I don t prove what I was expect to see: namely that , at the very early beginning stage of training I should not see any difference bet ween the regularized and non-regularized models and then, until a certain numner of epochs , more epochs I train  more I should need a  weight decay to compensate overfitting or just  generalize better. ( In my case was true only at the beginning, where wd=0 for epochs = 5, and then 0.0001 for epochs = 10 and 15)

in my opinion I think that the model isn t so much more complex respect to data and amount of data so  the "regularization/better generalization" happen during the training itsself and not from external parameters which are not the architecture itsself. following this reasoning, weight decay may have introduced an unnecessary constraint on the parameters,  limiting the model s ability to fit the right patterns

despite using the same tuning-like procedure and the cell output"Tuning weight decay for ShallowNet..." this is not intended as final hyperparameter selection for ShallowNet. It is only an exploratory test as mentioned. I forgot to change that print 

### Final training strategy
At this point I have one selected configuration for each architecture.
since are fixed, retrain each model on the full MNIST training set before evaluating on the test set.

This avoids wasting validation data in the final model

In [3]:
shallow_best_lr, shallow_best_bs, shallow_best_wd =  0.001, 32, 0.0
deep_best_lr, deep_best_bs, deep_best_wd = 0.001, 32, 0.0

In [8]:

print("Selected hyperparameters")
print("ShallowNet:", shallow_best_lr, shallow_best_bs)
print("DeepNet:   ", deep_best_lr, deep_best_bs, deep_best_wd)
#the shallow model in this experiments can t have a regularization parameter
# so the "shallow_best_wd" is set to 0
shallow_best_wd = 0.0


shallow_train_loader = make_loader(train_subset, batch_size=shallow_best_bs, shuffle=True, seed=SEED)
shallow_val_loader = make_loader(val_subset, batch_size=shallow_best_bs, shuffle=False, seed=SEED)

deep_train_loader = make_loader(train_subset, batch_size=deep_best_bs, shuffle=True, seed=SEED)
deep_val_loader = make_loader(val_subset, batch_size=deep_best_bs, shuffle=False, seed=SEED)


Selected hyperparameters
ShallowNet: 0.001 32
DeepNet:    0.001 32 0.0


In [ ]:
# Retrain each model on the full official training set before testing
print("Retraining final models on the full MNIST training set...")
test_loader = make_loader(test_data, batch_size=256, shuffle=False, seed=SEED)

shallow_epochs = 40
deep_epochs = 40

full_train_loader_shallow = make_loader(
    train_data,
    batch_size=shallow_best_bs,
    shuffle=True,
    seed=SEED,
)

full_train_loader_deep = make_loader(
    train_data,
    batch_size=deep_best_bs,
    shuffle=True,
    seed=SEED,
)

set_seed(SEED)
final_shallow_model = ShallowNet()
final_shallow_train_losses, final_shallow_train_accs, _, _ = train_model(
    final_shallow_model,
    full_train_loader_shallow,
    val_loader=None,
    lr=shallow_best_lr,
    epochs=shallow_epochs,
    weight_decay=shallow_best_wd,
)

set_seed(SEED)
final_deep_model = DeepNet()
final_deep_train_losses, final_deep_train_accs, _, _ = train_model(
    final_deep_model,
    full_train_loader_deep,
    val_loader=None,
    lr=deep_best_lr,
    epochs=deep_epochs,
    weight_decay=deep_best_wd,
)
# I will use them i the subsequent experiments (task 3 for the moment)
torch.save(final_deep_model.state_dict(), "weights_deep.pth")
torch.save(final_shallow_model.state_dict(), "weights_shallow.pth")



criterion = nn.CrossEntropyLoss()
test_loss_shallow, acc_shallow = evaluate(final_shallow_model, test_loader, criterion)
test_loss_deep, acc_deep = evaluate(final_deep_model, test_loader, criterion)

print("\nFINAL TEST ACCURACY")
print(f"ShallowNet: {acc_shallow:.2f}%")
print(f"DeepNet:    {acc_deep:.2f}%")

Retraining final models on the full MNIST training set...

FINAL TEST ACCURACY
ShallowNet: 98.00%
DeepNet:    98.44%


### Final conclusion


The hyperparameters choices: the learning rate is treated as the most critical parameter: the text shows that 0.1 is too high for shallownet and leads to lack of convergence, while the others are okay.
The batch size of 32 is a sensible middle ground between gradient estimation stability and computational cost. so I think that, at parity of validation error, (if the validation error is low and therefore also the convergence should be stable)we should choose the smaller one in order to save computation. adam optimizer is just a standard optimizer and I choose it for this.


The key point is we want to prove that:

DeepNet has a lots more parameters than ShallowNet, but despite this, it should generalizes better precisely because of regularization. this is what we would like to see


The validation and test results are not comparable because they do not derive from the same experimental procedure. the validation metrics used in hyperparameter selection were obtained over 10 epochs, while the final test was calculated after completely retraining the model for 40 epochs on the entire official training set. For this reason, the fact that the test accuracy is salmost higher the validation accuracy is not a strange behavior(best val acc=97.58% vs 98.00 for the shallow), although it s an unlikely behaviour since the test error should be much higher, but rather a consequence of the fact that the two numbers were produced under different conditions. A more correct procedure: after setting all three hyperparameters, retrain the model for 40 epochs on the training subset, evaluate it on the validation subset, and only then compare this final validation accuracy with the test accuracy.
In addition, it s very useful also compare the final test accuracy obtained from the final retraining on the full training set.

Now we can only compare the 2 final test accuracy and saying the Deepnet generalizes better, but it s a poor conclusion.

Another important observation:

I wanted to show that DeepNet generalizes better precisely because it's regularized. But I can't infer why it has a better test acc. Because between ShallowNet and DeepNet, the following changes simultaneously:
depth,
number of parameters,... not only reguolarization.So we can t knoe if the reason is precisely the regularization alone or the architecture itself.(or how much one influence onthe other one)

In the next experiment, I'll also train DeepNet without regularization and then do a final comparison between the three: ShallowNet, DeepNet without regularization, and DeepNet with regularization.

1. In the next experimnet I ll address the first point, to see how the training and validation curves behaviour

2. In the third experimnet I ll address the second point and I ll do the final conclusions

### Bad hyperparameters demonstration
The task also asks to show the effect of a poor hyperparameter choice.
A very large learning rate (0.1) is a good example as I already saw in the previous cell in the hyperparameter selection validation 

In [ ]:
print("Testing a bad learning rate (0.1) on ShallowNet")
set_seed(SEED)



bad_model1 = ShallowNet()

tl1, ta1, vl1, va1 = train_model(
    bad_model1,
    shallow_train_loader,
    shallow_val_loader,
    lr = 0.1,
    epochs = 15,
    weight_decay = 0.0,
)


print(f" final train loss: {tl1[-1]:.4f}, final train acc: {ta1[-1]:.2f}%")
print(f" final val loss:   {vl1[-1]:.4f}, final val acc:   {va1[-1]:.2f}%\n")

print("Testing a bad learning rate (0.1) on DeepNet")


set_seed(SEED)
bad_model2 = DeepNet()

tl2, ta2, vl2, va2 = train_model(
    bad_model2,
    deep_train_loader,
    deep_val_loader,
    lr = 0.1,
    epochs = 15,
    weight_decay = 0.0,
)

print(f"  final train loss: {tl2[-1]:.4f}, final train acc: {ta2[-1]:.2f}%")
print(f"  final val loss:   {vl2[-1]:.4f}, final val acc:   {va2[-1]:.2f}%")


Testing a bad learning rate (0.1) on ShallowNet
 final train loss: 2.3067, final train acc: 10.52%
 final val loss:   2.3120, final val acc:   11.33%

Testing a bad learning rate (0.1) on DeepNet
  final train loss: 0.3266, final train acc: 90.99%
  final val loss:   0.1142, final val acc:   97.31%



The results show that shallowNet does not converge, since both training and validation accuracy stay at random-guess level. This means that the poor hyperparameter choice strongly affects the simpler model.

On the other hand, we haven t found so far a bad parameter for the deep model